# SPRT‑TANDEM Real‑Time Atrial Fibrillation Detection (PhysioNet/CinC 2017)

**Raw ECG stream – Normal vs AF – Early stopping via definitive online Wald SPRT**

## 1. Imports and Device

In [ ]:
import os, math, time, random, warnings, zipfile, io
import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             confusion_matrix, accuracy_score,
                             recall_score, precision_score,
                             ConfusionMatrixDisplay)
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import kagglehub

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Download and Load Dataset

In [ ]:
path = kagglehub.dataset_download("awsaf49/physionet-cinc-2017-dataset")
data_dir = path + '/training2017/'

ref_df = pd.read_csv(data_dir + 'REFERENCE.csv', header=None, names=['fname', 'label'])
ref_df = ref_df[ref_df['label'].isin(['N', 'A'])]  # Normal and AF only
ref_df['binary_label'] = (ref_df['label'] == 'A').astype(int)
print(f"Total Normal + AF recordings: {len(ref_df)}")
print(ref_df['label'].value_counts())

recordings = []
labels = []
for _, row in tqdm(ref_df.iterrows(), total=len(ref_df), desc="Loading .mat files"):
    mat = sio.loadmat(data_dir + row['fname'] + '.mat')
    ecg = mat['val'].flatten().astype(np.float32)
    recordings.append(ecg)
    labels.append(row['binary_label'])

print(f"Loaded {len(recordings)} recordings")

## 3. Data Display

In [ ]:
print('Class distribution:')
print(pd.Series(labels).value_counts())

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
np.random.seed(1)
for i, ax in enumerate(axes.flatten()):
    idx = np.random.choice(len(recordings))
    ax.plot(recordings[idx][:3000])
    ax.set_title(f"Label: {'AF' if labels[idx]==1 else 'Normal'}")
plt.tight_layout()
plt.show()

## 4. Slice Recordings and Build Sequences

In [ ]:
SLICE_LEN = 75
SLICE_STRIDE = 75
T = 120

def slice_recording(ecg):
    slices = []
    for start in range(0, len(ecg) - SLICE_LEN + 1, SLICE_STRIDE):
        slices.append(ecg[start:start + SLICE_LEN])
    return np.array(slices)

all_sequences, all_labels = [], []
for ecg, lbl in zip(recordings, labels):
    slices = slice_recording(ecg)
    if len(slices) < T:
        continue
    all_sequences.append(slices[:T])
    all_labels.append(lbl)

X_all = np.array(all_sequences, dtype=np.float32)
y_all = np.array(all_labels, dtype=np.int64)
print(f'Total sequences: {X_all.shape}, labels: {y_all.shape}')
print(f'Class balance: {np.bincount(y_all)}')

# Chronological split
n_total = len(X_all)
train_end = int(n_total * 0.7)
val_end = int(n_total * 0.85)
X_train, y_train = X_all[:train_end], y_all[:train_end]
X_val, y_val = X_all[train_end:val_end], y_all[train_end:val_end]
X_test, y_test = X_all[val_end:], y_all[val_end:]
X_test_raw = X_test.copy()

# Scale per slice
all_train_slices = X_train.reshape(-1, SLICE_LEN)
scaler = StandardScaler().fit(all_train_slices)

def scale_sequences(X):
    orig = X.shape
    X_flat = X.reshape(-1, SLICE_LEN)
    return scaler.transform(X_flat).reshape(orig)

X_train = scale_sequences(X_train)
X_val = scale_sequences(X_val)
X_test = scale_sequences(X_test)

## 5. Phase 1 – CNN Feature Extractor

In [ ]:
FEAT_DIM = 128

class BetterSliceCNN(nn.Module):
    def __init__(self, feat_dim=128, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, 7, padding=3, stride=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.res1 = self._make_res_block(32, 64, stride=2)
        self.res2 = self._make_res_block(64, 128, stride=2)
        self.res3 = self._make_res_block(128, feat_dim, stride=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(feat_dim, num_classes)

    @staticmethod
    def _make_res_block(in_ch, out_ch, stride):
        return nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 5, stride=stride, padding=2),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
            nn.Conv1d(out_ch, out_ch, 5, stride=1, padding=2),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(),
        )

    def forward(self, x):
        x = x.view(-1, 1, SLICE_LEN)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.res1(x)
        x = self.res2(x)
        x = self.res3(x)
        feats = self.pool(x).squeeze(-1)
        logits = self.classifier(feats)
        return logits, feats

# Balance training set
all_train_slices_flat = X_train.reshape(-1, SLICE_LEN)
train_labels_flat = np.repeat(y_train, T)
class0_idx = np.where(train_labels_flat == 0)[0]
class1_idx = np.where(train_labels_flat == 1)[0]
min_len = min(len(class0_idx), len(class1_idx))
sel0 = np.random.RandomState(42).choice(class0_idx, min_len, replace=False)
sel1 = np.random.RandomState(42).choice(class1_idx, min_len, replace=False)
bal_idx = np.concatenate([sel0, sel1]); np.random.shuffle(bal_idx)
train_slices_bal = all_train_slices_flat[bal_idx]
train_labels_bal = train_labels_flat[bal_idx]

train_X = torch.tensor(train_slices_bal, dtype=torch.float32)
train_Y = torch.tensor(train_labels_bal, dtype=torch.long)
phase1_loader = DataLoader(TensorDataset(train_X, train_Y), batch_size=256, shuffle=True)

cnn = BetterSliceCNN(FEAT_DIM).to(device)
opt = optim.Adam(cnn.parameters(), lr=1e-3, weight_decay=0.0)
criterion = nn.CrossEntropyLoss()

best_val_bacc, patience_counter = 0.0, 0
for epoch in range(100):
    cnn.train(); total_loss = 0
    for xb, yb in phase1_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits, _ = cnn(xb)
        loss = criterion(logits, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    val_slices = X_val.reshape(-1, SLICE_LEN)
    val_X = torch.tensor(val_slices, dtype=torch.float32).to(device)
    with torch.no_grad():
        preds = cnn(val_X)[0].argmax(1).cpu().numpy()
    val_bacc = balanced_accuracy_score(np.repeat(y_val, T), preds)
    if val_bacc - best_val_bacc > 0.001:
        best_val_bacc = val_bacc; patience_counter = 0
        torch.save(cnn.state_dict(), 'best_cnn.pth')
    else:
        patience_counter += 1
    if (epoch + 1) % 5 == 0:
        print(f'Phase1 Epoch {epoch+1:3d}: val bacc={val_bacc:.4f} (best={best_val_bacc:.4f})')
    if patience_counter >= 10:
        print('Early stopping')
        break
cnn.load_state_dict(torch.load('best_cnn.pth')); cnn.eval()
print(f'Best Phase1 val bacc: {best_val_bacc:.4f}')

## 6. Extract Features

In [ ]:
def extract_features(model, X_arr, batch_size=256):
    feats = []
    for i in range(0, len(X_arr), batch_size):
        xb = torch.tensor(X_arr[i:i+batch_size], dtype=torch.float32).to(device)
        with torch.no_grad():
            _, f = model(xb)
        feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)

train_feats = extract_features(cnn, X_train.reshape(-1, SLICE_LEN)).reshape(X_train.shape[0], T, FEAT_DIM)
val_feats = extract_features(cnn, X_val.reshape(-1, SLICE_LEN)).reshape(X_val.shape[0], T, FEAT_DIM)
test_feats = extract_features(cnn, X_test.reshape(-1, SLICE_LEN)).reshape(X_test.shape[0], T, FEAT_DIM)

train_set = TensorDataset(torch.tensor(train_feats, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_set = TensorDataset(torch.tensor(val_feats, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_set = TensorDataset(torch.tensor(test_feats, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
X_test_seq_raw = X_test_raw

## 7. SPRT‑TANDEM Integrator

In [ ]:
def B2Bsqrt(x, alpha=1.0):
    a = torch.tensor(alpha, device=x.device, dtype=x.dtype)
    return torch.sign(x) * (torch.sqrt(a + torch.abs(x)) - torch.sqrt(a))

class StandardLSTM(nn.Module):
    def __init__(self, feat_dim, hidden_size, out_activation='B2Bsqrt'):
        super().__init__()
        self.feat_dim, self.hidden_size, self.out_activation = feat_dim, hidden_size, out_activation
        self.W_i = nn.Parameter(torch.Tensor(feat_dim, hidden_size))
        self.U_i = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_i = nn.Parameter(torch.Tensor(hidden_size))
        self.W_f = nn.Parameter(torch.Tensor(feat_dim, hidden_size))
        self.U_f = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_f = nn.Parameter(torch.Tensor(hidden_size))
        self.W_c = nn.Parameter(torch.Tensor(feat_dim, hidden_size))
        self.U_c = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_c = nn.Parameter(torch.Tensor(hidden_size))
        self.W_o = nn.Parameter(torch.Tensor(feat_dim, hidden_size))
        self.U_o = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_o = nn.Parameter(torch.Tensor(hidden_size))
        self.init_weights()

    def init_weights(self):
        for w in [self.W_i, self.W_f, self.W_c, self.W_o, self.U_i, self.U_f, self.U_c, self.U_o]:
            nn.init.xavier_uniform_(w)
        for b in [self.b_i, self.b_f, self.b_c, self.b_o]:
            nn.init.constant_(b, 0)

    def forward(self, x):
        batch, steps, _ = x.shape
        h = torch.zeros(batch, self.hidden_size, device=x.device)
        c = torch.zeros(batch, self.hidden_size, device=x.device)
        outs = []
        for t in range(steps):
            xt = x[:, t, :]
            i_t = torch.sigmoid(xt @ self.W_i + h @ self.U_i + self.b_i)
            f_t = torch.sigmoid(xt @ self.W_f + h @ self.U_f + self.b_f)
            o_t = torch.sigmoid(xt @ self.W_o + h @ self.U_o + self.b_o)
            c_tilde = B2Bsqrt(xt @ self.W_c + h @ self.U_c + self.b_c)
            c = f_t * c + i_t * c_tilde
            h = o_t * B2Bsqrt(c) if self.out_activation == 'B2Bsqrt' else o_t * torch.tanh(c)
            outs.append(h.unsqueeze(1))
        return torch.cat(outs, dim=1)

def calc_llrs(logits_concat, order_sprt, time_steps, num_classes):
    batch, eff_T, win_len, nc = logits_concat.shape
    logits1 = logits_concat.unsqueeze(-1)
    logits2 = logits_concat.unsqueeze(-2)
    llr_list = []
    if order_sprt == 0:
        lr = logits1[:, :, 0, :, :] - logits2[:, :, 0, :, :]
        llrs = torch.cumsum(lr, dim=1)
    else:
        for t in range(time_steps):
            if t < order_sprt + 1:
                lr = logits1[:, 0, t, :, :] - logits2[:, 0, t, :, :]
            else:
                idx = t - order_sprt
                lr1 = logits1[:, :idx, order_sprt, :, :] - logits2[:, :idx, order_sprt, :, :]
                lr1 = torch.sum(lr1, dim=1)
                if idx > 1:
                    lr2 = logits1[:, 1:idx, order_sprt - 1, :, :] - logits2[:, 1:idx, order_sprt - 1, :, :]
                    lr2 = torch.sum(lr2, dim=1)
                else:
                    lr2 = torch.zeros_like(lr1)
                lr = lr1 - lr2
            llr_list.append(lr.unsqueeze(1))
        llrs = torch.cat(llr_list, dim=1)
    tri = torch.ones_like(llrs)
    llrs = llrs - 1e-12 * (torch.triu(tri) - torch.tril(tri))
    return llrs

def restore_lost_significance(llrs):
    tri = torch.ones_like(llrs)
    return llrs - 1e-10 * (torch.triu(tri) - torch.tril(tri))

def multiplet_ce_flat(logits_flat, labels_slice):
    logits = logits_flat.permute(1, 0, 2).reshape(-1, logits_flat.shape[-1])
    labels = labels_slice.repeat(logits_flat.shape[1])
    return F.cross_entropy(logits, labels)

def lsel_loss(llrs, labels):
    bs, T_, nc, _ = llrs.shape
    labels_oh = F.one_hot(labels, nc).float().view(bs, 1, nc, 1)
    llrs_masked = llrs * labels_oh
    llrs_vec = torch.sum(llrs_masked, dim=2)
    llrs_flat = llrs_vec.reshape(-1, nc)
    minllr, _ = torch.min(llrs_flat, dim=1, keepdim=True)
    llrs_stable = llrs_flat - minllr
    loss = -minllr.squeeze(1) + torch.log(torch.sum(torch.exp(-llrs_stable), dim=1) + 1e-12)
    return loss.mean()

ORDER = 5
NUM_CLASSES = 2
WIDTH_LSTM = 64
DROPOUT = 0.1

class SPRTTANDEMIntegrator(nn.Module):
    def __init__(self, feat_dim, num_classes, order_sprt, hidden_size,
                 time_steps=T, dropout=DROPOUT, use_pe=True):
        super().__init__()
        self.order_sprt = order_sprt
        self.time_steps = time_steps
        self.num_classes = num_classes
        self.use_pe = use_pe
        if use_pe:
            self.pos_encoding = nn.Parameter(torch.randn(order_sprt + 1, feat_dim))
        self.lstm = StandardLSTM(feat_dim, hidden_size, out_activation='B2Bsqrt')
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        bs, T_full, F = x.shape
        win_len = self.order_sprt + 1
        eff_T = T_full - self.order_sprt
        windows = []
        for i in range(eff_T):
            windows.append(x[:, i:i + win_len, :])
        x_windows = torch.stack(windows, dim=1)
        if self.use_pe:
            x_windows = x_windows + self.pos_encoding.view(1, 1, win_len, F)
        x_flat = x_windows.reshape(bs * eff_T, win_len, F)
        lstm_out = self.lstm(x_flat)
        lstm_out = self.dropout(lstm_out)
        logits_flat = self.fc(lstm_out)
        logits_struct = logits_flat.view(bs, eff_T, win_len, self.num_classes)
        return logits_flat, logits_struct, (bs, eff_T)

# Training
LR = 1e-4
W_MCE = 0.6
W_LSEL = 0.3
MAX_NORM = 50000.0

integrator = SPRTTANDEMIntegrator(FEAT_DIM, NUM_CLASSES, ORDER, WIDTH_LSTM,
                                   time_steps=T, use_pe=True).to(device)
optimizer = optim.Adam(integrator.parameters(), lr=LR, weight_decay=0.0)

best_val_bacc2, patience_cnt2 = 0.0, 0
for epoch in range(500):
    integrator.train()
    tot_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits_flat, logits_struct, (bs, eff_T) = integrator(xb)
        llrs = calc_llrs(logits_struct, ORDER, T, NUM_CLASSES)
        y_slice = yb.repeat(eff_T)
        mce = multiplet_ce_flat(logits_flat, y_slice)
        lsel = lsel_loss(llrs, yb)
        loss = W_MCE * mce + W_LSEL * lsel
        loss.backward()
        torch.nn.utils.clip_grad_norm_(integrator.parameters(), MAX_NORM)
        optimizer.step()
        tot_loss += loss.item()

    integrator.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb_v, yb_v in val_loader:
            xb_v, yb_v = xb_v.to(device), yb_v.to(device)
            _, logits_v, _ = integrator(xb_v)
            llrs_v = calc_llrs(logits_v, ORDER, T, NUM_CLASSES)
            lr_last = llrs_v[:, -1, 1, 0]
            pred = (lr_last > 0).long()
            all_preds.append(pred.cpu()); all_labels.append(yb_v.cpu())
    val_bacc = balanced_accuracy_score(torch.cat(all_labels).numpy(),
                                       torch.cat(all_preds).numpy())
    if val_bacc - best_val_bacc2 > 0.001:
        best_val_bacc2 = val_bacc; patience_cnt2 = 0
        torch.save(integrator.state_dict(), 'best_integrator.pth')
    else:
        patience_cnt2 += 1
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}: loss={tot_loss/len(train_loader):.4f}, val bacc={val_bacc:.4f} (best={best_val_bacc2:.4f})')
    if patience_cnt2 >= 80:
        print('Early stopping')
        break
integrator.load_state_dict(torch.load('best_integrator.pth')); integrator.eval()
print(f'Best integrator val bacc: {best_val_bacc2:.4f}')

## 8. Baseline – Fixed‑Length LSTM (Trained per Observation Length)

In [ ]:
class FixedLSTM(nn.Module):
    def __init__(self, feat_dim, hidden_size, num_classes, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(feat_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))


def build_sequences_of_length(X_arr, y_arr, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X_arr) - seq_len + 1):
        X_seq.append(X_arr[i:i + seq_len])
        y_seq.append(y_arr[i + seq_len - 1])
    return np.array(X_seq), np.array(y_seq)


def train_fixed_lstm_at_length(k):
    X_tr_seq, y_tr_seq = build_sequences_of_length(X_train_scaled, y_train, k)
    X_val_seq_k, y_val_seq_k = build_sequences_of_length(X_val_scaled, y_val, k)

    tr_feats = extract_features(cnn, X_tr_seq.reshape(-1, SLICE_LEN)).reshape(-1, k, FEAT_DIM)
    val_feats_k = extract_features(cnn, X_val_seq_k.reshape(-1, SLICE_LEN)).reshape(-1, k, FEAT_DIM)

    train_ldr = DataLoader(TensorDataset(torch.tensor(tr_feats, dtype=torch.float32),
                                         torch.tensor(y_tr_seq, dtype=torch.long)),
                           batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_ldr = DataLoader(TensorDataset(torch.tensor(val_feats_k, dtype=torch.float32),
                                       torch.tensor(y_val_seq_k, dtype=torch.long)),
                         batch_size=BATCH_SIZE, shuffle=False)

    model = FixedLSTM(FEAT_DIM, WIDTH_LSTM, NUM_CLASSES).to(device)
    opt_base = optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)
    best_bacc = 0.0
    patience = 0
    for epoch in range(150):
        model.train()
        for xb, yb in train_ldr:
            xb, yb = xb.to(device), yb.to(device)
            opt_base.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)
            opt_base.step()
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for xb, yb in val_ldr:
                pred = model(xb.to(device)).argmax(1).cpu()
                all_preds.append(pred)
                all_labels.append(yb)
        val_bacc = balanced_accuracy_score(torch.cat(all_labels).numpy(),
                                           torch.cat(all_preds).numpy())
        if val_bacc - best_bacc > 0.0005:
            best_bacc = val_bacc; patience = 0
            torch.save(model.state_dict(), f'best_baseline_len{k}.pth')
        else:
            patience += 1
        if patience >= 40:
            break
    model.load_state_dict(torch.load(f'best_baseline_len{k}.pth'))
    model.eval()
    return model, best_bacc


X_train_scaled = X_train.reshape(-1, SLICE_LEN)
X_val_scaled = X_val.reshape(-1, SLICE_LEN)
X_test_scaled = X_test.reshape(-1, SLICE_LEN)

horizons = [20, 40, 60, 80, 100, 120]
baseline_models = {}
baseline_baccs = {}
for k in horizons:
    print(f'Training baseline for length {k}...')
    model, bacc = train_fixed_lstm_at_length(k)
    baseline_models[k] = model
    baseline_baccs[k] = bacc
    print(f'  val bacc = {bacc:.4f}')

## 9. Definitive Online Inference for SPRT‑TANDEM

In [ ]:
def online_inference(model, feature_extractor, scaler_obj, X_stream, y_stream,
                     th_a1, th_a0, order=ORDER):
    if X_stream.ndim == 2:
        X_stream = X_stream[np.newaxis, ...]

    n_seq, T_, _ = X_stream.shape
    model.eval()
    feature_extractor.eval()
    pos_enc = model.pos_encoding.detach()
    mean_t = torch.tensor(scaler_obj.mean_, dtype=torch.float32, device=device)
    std_t  = torch.tensor(scaler_obj.scale_, dtype=torch.float32, device=device)

    all_preds = np.empty(n_seq, dtype=np.int64)
    all_hts   = np.empty(n_seq, dtype=np.int64)

    for seq_idx in range(n_seq):
        buffer = []
        total_llr = None
        decision = None
        stop_t = None

        for t in range(T_):
            # 1. Scale and extract feature for current observation
            x_raw = X_stream[seq_idx, t]
            x_tensor = torch.tensor(x_raw, dtype=torch.float32, device=device)
            x_tensor = (x_tensor - mean_t) / (std_t + 1e-8)
            with torch.no_grad():
                _, feat = feature_extractor(x_tensor.unsqueeze(0))
            buffer.append(feat.cpu().numpy()[0])
            if len(buffer) > order + 1:
                buffer.pop(0)

            # 2. Process sliding window when full
            if len(buffer) == order + 1:
                buf_tensor = torch.tensor(buffer, dtype=torch.float32, device=device).unsqueeze(0)
                buf_tensor = buf_tensor + pos_enc.unsqueeze(0)
                with torch.no_grad():
                    lstm_out = model.lstm(buf_tensor)
                    logits = model.fc(lstm_out)          # (1, order+1, 2)

                if total_llr is None:
                    # ----- First full window -----
                    for s in range(order + 1):
                        λ_s = float(logits[0, s, 1] - logits[0, s, 0])
                        if λ_s >= th_a1:
                            decision = 1; stop_t = s + 1; break
                        elif λ_s <= -th_a0:
                            decision = 0; stop_t = s + 1; break
                    if decision is not None:
                        break
                    total_llr = float(logits[0, order, 1] - logits[0, order, 0])
                else:
                    # ----- Subsequent windows: TANDEM increment -----
                    lr_N1 = float(logits[0, -1, 1] - logits[0, -1, 0])
                    lr_N  = float(logits[0, -2, 1] - logits[0, -2, 0])
                    total_llr += (lr_N1 - lr_N)
                    if total_llr >= th_a1:
                        decision = 1; stop_t = t + 1; break
                    elif total_llr <= -th_a0:
                        decision = 0; stop_t = t + 1; break

        # 3. Forced decision if threshold never crossed
        if decision is None:
            decision = 1 if total_llr > 0 else 0
            stop_t = T_

        all_preds[seq_idx] = decision
        all_hts[seq_idx] = stop_t

    return all_preds, all_hts

## 10. Speed‑Accuracy Trade‑Off (1000 Log‑Spaced Thresholds, Full MSPRT)

In [ ]:
def evaluate_sprt_tandem_sat(model, loader, num_thresh=1000):
    model.eval()
    all_llrs, all_labels = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        _, logits, (bs, eff_T) = model(xb)
        llrs = calc_llrs(logits, ORDER, T, NUM_CLASSES)
        all_llrs.append(llrs.cpu()); all_labels.append(yb)
    llrs_t = torch.cat(all_llrs, dim=0)
    labels_t = torch.cat(all_labels, dim=0)
    llrs_t = restore_lost_significance(llrs_t)

    n, Tclip, nc = llrs_t.shape[0], llrs_t.shape[1], llrs_t.shape[2]
    dev = llrs_t.device
    abs_vals = llrs_t.abs()
    min_abs = abs_vals[abs_vals > 1e-12].min().item()
    max_abs = abs_vals.max().item()
    threshs = torch.exp(torch.linspace(math.log(min_abs), math.log(max_abs), num_thresh))

    eye = torch.eye(nc, device=dev)
    mhts, baccs = [], []
    for thr in threshs:
        thresh_mtx = thr * (1 - eye).view(1, 1, 1, nc, nc).repeat(1, 1, Tclip, 1, 1)
        scores_full = llrs_t.unsqueeze(0) - thresh_mtx
        scores = torch.min(scores_full, dim=-1)[0]
        preds_all = torch.sign(scores) + 1
        last_llr = llrs_t[:, -1:, :, :]
        last_scores = torch.min(last_llr, dim=-1)[0]
        preds_last = torch.sign(last_scores) + 1
        preds_last = preds_last.unsqueeze(0)
        preds_trunc = torch.cat([preds_all[:, :, :-1, :], preds_last], dim=2)
        mask = torch.arange(Tclip, 0, -1, dtype=torch.float32, device=dev).view(1, 1, Tclip, 1)
        masked = preds_trunc * mask
        hit_idx = torch.max(masked, dim=2)[0]
        _, pred_class = torch.max(hit_idx, dim=2)
        hit_time = Tclip - torch.max(hit_idx, dim=2)[0] + 1

        labels = labels_t
        mht = hit_time.float().mean().item()
        tp = ((pred_class == 1) & (labels == 1)).sum().item()
        tn = ((pred_class == 0) & (labels == 0)).sum().item()
        pos = (labels == 1).sum().item(); neg = (labels == 0).sum().item()
        bacc = ((tp / pos if pos else 0) + (tn / neg if neg else 0)) / 2.0
        mhts.append(mht); baccs.append(bacc)

    return threshs.numpy(), np.array(mhts), np.array(baccs)


alpha, beta = 0.05, 0.01
a_wald1 = math.log((1 - beta) / alpha)
a_wald0 = math.log((1 - alpha) / beta)
print(f'Wald thresholds: a1={a_wald1:.3f}, a0={a_wald0:.3f}')

threshs_val, mhts_val, baccs_val = evaluate_sprt_tandem_sat(integrator, val_loader)
a_opt = threshs_val[np.argmax(baccs_val)]
print(f'Optimal threshold (validation): {a_opt:.3f}')

preds_opt, hts_opt = online_inference(integrator, cnn, scaler,
                                       X_test_seq_raw, y_test, a_opt, a_opt)
preds_wald, hts_wald = online_inference(integrator, cnn, scaler,
                                         X_test_seq_raw, y_test, a_wald1, a_wald0)

# Baseline evaluation for each observation length
baseline_results = {}
for k, model_bl in baseline_models.items():
    X_test_seq_k, y_test_seq_k = build_sequences_of_length(X_test_scaled, y_test, k)
    test_feats_k = extract_features(cnn, X_test_seq_k.reshape(-1, SLICE_LEN)).reshape(-1, k, FEAT_DIM)
    test_ldr = DataLoader(TensorDataset(torch.tensor(test_feats_k, dtype=torch.float32),
                                        torch.tensor(y_test_seq_k, dtype=torch.long)),
                          batch_size=BATCH_SIZE, shuffle=False)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in test_ldr:
            pred = model_bl(xb.to(device)).argmax(1).cpu()
            all_preds.append(pred); all_labels.append(yb)
    preds_bl = torch.cat(all_preds).numpy()
    labs_bl = torch.cat(all_labels).numpy()
    baseline_results[k] = (balanced_accuracy_score(labs_bl, preds_bl), f1_score(labs_bl, preds_bl))

threshs_test, mhts_test, baccs_test = evaluate_sprt_tandem_sat(integrator, test_loader)

## 11. Metrics and Comparison Table

In [ ]:
def compute_metrics(y_true, y_pred, hts):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    return {
        'Bal.Acc': balanced_accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'Sens': sens, 'Spec': spec, 'Prec': prec,
        'Mean HT': np.mean(hts),
        'Time Saved %': (1 - np.mean(hts) / T) * 100
    }

metrics_opt = compute_metrics(y_test, preds_opt, hts_opt)
metrics_wald = compute_metrics(y_test, preds_wald, hts_wald)

# Full‑length baseline
k_full = T
X_test_seq_full, y_test_seq_full = build_sequences_of_length(X_test_scaled, y_test, k_full)
test_feats_full = extract_features(cnn, X_test_seq_full.reshape(-1, SLICE_LEN)).reshape(-1, k_full, FEAT_DIM)
test_ldr_full = DataLoader(TensorDataset(torch.tensor(test_feats_full, dtype=torch.float32),
                                         torch.tensor(y_test_seq_full, dtype=torch.long)),
                           batch_size=BATCH_SIZE, shuffle=False)
all_preds_base, all_labs_base = [], []
with torch.no_grad():
    for xb, yb in test_ldr_full:
        pred = baseline_models[k_full](xb.to(device)).argmax(1).cpu()
        all_preds_base.append(pred); all_labs_base.append(yb)
preds_base = torch.cat(all_preds_base).numpy()
labs_base = torch.cat(all_labs_base).numpy()
base_bacc = balanced_accuracy_score(labs_base, preds_base)
base_f1 = f1_score(labs_base, preds_base)
base_sens = recall_score(labs_base, preds_base)
tn, fp, fn, tp = confusion_matrix(labs_base, preds_base).ravel()
base_spec = tn / (tn + fp) if (tn + fp) > 0 else 0
base_prec = tp / (tp + fp) if (tp + fp) > 0 else 0
metrics_base = {'Bal.Acc': base_bacc, 'F1': base_f1, 'Sens': base_sens,
                'Spec': base_spec, 'Prec': base_prec, 'Mean HT': float(T), 'Time Saved %': 0.0}

df_metrics = pd.DataFrame([metrics_opt, metrics_wald, metrics_base])
df_metrics.index = ['SPRT-TANDEM (opt)', 'SPRT-TANDEM (Wald)', 'Baseline (full 120)']
print(df_metrics.to_string(float_format='%.3f'))

## 12. Visualisations

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(mhts_test, baccs_test, 'b-o', markersize=2, label='SPRT-TANDEM SAT')
for k, (bacc_k, _) in baseline_results.items():
    plt.scatter(k, bacc_k, color='orange', marker='X', s=80, zorder=5)
plt.scatter([T], [metrics_base['Bal.Acc']], color='r', marker='X', s=120, label='Baseline 120')
plt.scatter([metrics_opt['Mean HT']], [metrics_opt['Bal.Acc']],
            color='g', marker='D', s=120, label='Online opt')
plt.scatter([metrics_wald['Mean HT']], [metrics_wald['Bal.Acc']],
            color='m', marker='s', s=100, label='Online Wald')
plt.xlabel('Mean hitting time (slices)')
plt.ylabel('Balanced accuracy')
plt.title('Speed-Accuracy Trade-Off')
plt.legend(); plt.grid(True); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(hts_opt, bins=30, alpha=0.7, label='Optimal threshold')
axes[0].axvline(np.mean(hts_opt), color='k', linestyle='--')
axes[0].set_xlabel('Hitting time'); axes[0].set_title('Online (opt)')
axes[1].hist(hts_wald, bins=30, alpha=0.7, label='Wald threshold')
axes[1].axvline(np.mean(hts_wald), color='k', linestyle='--')
axes[1].set_xlabel('Hitting time'); axes[1].set_title('Online (Wald)')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
ConfusionMatrixDisplay.from_predictions(y_test, preds_opt, ax=axes[0], colorbar=False)
axes[0].set_title('Optimal SPRT')
ConfusionMatrixDisplay.from_predictions(y_test, preds_wald, ax=axes[1], colorbar=False)
axes[1].set_title('Wald SPRT')
ConfusionMatrixDisplay.from_predictions(labs_base, preds_base, ax=axes[2], colorbar=False)
axes[2].set_title('Baseline 120')
plt.tight_layout(); plt.show()

test_llr_list = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        _, logits, _ = integrator(xb)
        llrs = calc_llrs(logits, ORDER, T, NUM_CLASSES)
        test_llr_list.append(llrs[:, :, 1, 0].cpu().numpy())
llr_all = np.concatenate(test_llr_list, axis=0)
plt.figure(figsize=(10, 4))
for i in range(5):
    plt.plot(range(T), llr_all[i], label=f"True:{y_test[i]}")
plt.axhline(y=a_opt, color='g', linestyle='--', label='Opt thresh')
plt.axhline(y=-a_opt, color='g', linestyle='--')
plt.xlabel('Slice'); plt.ylabel('LLR (log p1/p0)')
plt.title('LLR trajectories (first 5 test examples)')
plt.legend(); plt.grid(True); plt.show()